In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle


In [2]:
data = pd.read_csv('dataset/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
encoder = OneHotEncoder(drop=None,sparse_output = False)
encoded_df=encoder.fit_transform(data[['Gender', 'Geography']])

In [5]:
encoded_df = pd.DataFrame(encoded_df,columns=encoder.get_feature_names_out(['Gender','Geography']))
encoded_df

,Gender_Female,Gender_Male,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,1.0,0.0,0.0
1,1.0,0.0,0.0,0.0,1.0
2,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
9995,0.0,1.0,1.0,0.0,0.0
9996,0.0,1.0,1.0,0.0,0.0
9997,1.0,0.0,1.0,0.0,0.0
9998,0.0,1.0,0.0,1.0,0.0


In [6]:
data = pd.concat([data.drop(['Geography','Gender'],axis=1),encoded_df],axis=1)

In [7]:
data.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Gender_Female,Gender_Male,Geography_France,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,1.0,0.0,0.0
1,608,41,1,83807.86,1,0,1,112542.58,0,1.0,0.0,0.0,0.0,1.0
2,502,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,1.0,0.0,0.0
3,699,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,1.0,0.0,0.0
4,850,43,2,125510.82,1,1,1,79084.10,0,1.0,0.0,0.0,0.0,1.0


In [8]:
X=data.drop(['EstimatedSalary'],axis=1)
y=data['EstimatedSalary']

In [9]:
X.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Gender_Female,Gender_Male,Geography_France,Geography_Germany,Geography_Spain
0,619,42,2,0.00,1,1,1,1,1.0,0.0,1.0,0.0,0.0
1,608,41,1,83807.86,1,0,1,0,1.0,0.0,0.0,0.0,1.0
2,502,42,8,159660.80,3,1,0,1,1.0,0.0,1.0,0.0,0.0
3,699,39,1,0.00,2,0,0,0,1.0,0.0,1.0,0.0,0.0
4,850,43,2,125510.82,1,1,1,0,1.0,0.0,0.0,0.0,1.0


In [10]:
y.head()

0    101348.88
1    112542.58
2    113931.57
3     93826.63
4     79084.10
Name: EstimatedSalary, dtype: float64

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
X_train

array([[ 0.21835119,  1.91661905, -1.73168869, ...,  1.00053348,
        -0.57776083, -0.57735027],
       [ 2.05728037,  0.20210899,  1.04174968, ..., -0.99946681,
         1.73082   , -0.57735027],
       [ 0.75860157, -0.75039661,  0.34839008, ...,  1.00053348,
        -0.57776083, -0.57735027],
       ...,
       [ 0.86249588, -0.08364269, -1.3850089 , ...,  1.00053348,
        -0.57776083, -0.57735027],
       [ 0.15601461,  0.3926101 ,  1.04174968, ...,  1.00053348,
        -0.57776083, -0.57735027],
       [ 0.46769752,  1.15461458, -1.3850089 , ..., -0.99946681,
         1.73082   , -0.57735027]], shape=(7500, 13))

In [14]:
with open('artifacts/encoder_reg.pkl','wb') as file:
    pickle.dump(encoder,file)

with open('artifacts/scaler_reg.pkl','wb') as file:
    pickle.dump(scaler,file)

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [16]:
model = Sequential([
            Input(shape=(X_train.shape[1],)),
            Dense(64,activation='relu'), ##HL 1
            Dense(32,activation='relu'),  ##HL 2
            Dense(1) ##Output
])

In [17]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.MeanAbsoluteError()

In [18]:
model.compile(optimizer=opt,loss=loss,metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,009 (11.75 KB)

 Trainable params: 3,009 (11.75 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
from keras.callbacks import EarlyStopping,TensorBoard
import datetime

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)


In [20]:
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [21]:
history = model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 81851.4453 - mae: 81851.4453 - val_loss: 50987.8828 - val_mae: 50987.8828
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 50316.0391 - mae: 50316.0391 - val_loss: 50332.8008 - val_mae: 50332.8008
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49908.1680 - mae: 49908.1680 - val_loss: 50288.5391 - val_mae: 50288.5391
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49793.4688 - mae: 49793.4688 - val_loss: 50206.2852 - val_mae: 50206.2852
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49685.1055 - mae: 49685.1055 - val_loss: 50106.1055 - val_mae: 50106.1055
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49639.5273 - mae: 49639.5273 - val_loss: 50240.8320 - val_mae: 50240.8320
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49547.5547 - mae: 49547.5547 - val_loss: 50173.9297 - val_mae: 50173.9297
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - 

In [26]:
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f"MAE : {test_mae}")

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50094.7344 - mae: 50094.7344
MAE : 50094.734375


In [28]:
model.save('artifacts/regressionmodel.h5')